# Recursive Sketch hyperparameter search across OpenML-CC18

This notebook is intentionally thin: OpenML loading, candidate construction, the random-search loop, result summaries, and visualizations live in `openml_hyperparameter_search.py`. Configure the experiment below, run the search, and then render the reports.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

import pandas as pd

notebooks_dir = Path.cwd() / 'notebooks'
if not (notebooks_dir / 'openml_hyperparameter_search.py').exists():
    notebooks_dir = Path.cwd()
if str(notebooks_dir) not in sys.path:
    sys.path.insert(0, str(notebooks_dir))

from openml_hyperparameter_search import (
    best_configuration,
    best_scores_by_dataset,
    plot_marginalized_performance,
    plot_pairwise_performance,
    plot_top_candidates,
    run_search,
)

In [ ]:
# Edit this cell to configure the OpenML search.
DATASET_SEED = 4279
SEARCH_SEED = 177
DATASET_NAMES = ['kr-vs-kp', 'letter']

SMOKE_TEST = True
DATASET_COUNT = 3 if SMOKE_TEST else 8
MAX_DATASET_SIZE = 300 if SMOKE_TEST else 1000
REPETITIONS = tuple(range(5)) if SMOKE_TEST else tuple(range(10))
N_SEARCH_ITER = 10 if SMOKE_TEST else 100
TEST_SIZE = 0.30
N_JOBS = 1
OUTPUT_CLASSIFIER_N_ESTIMATORS = 100

# RecursiveSketch parameters plus model choices; each list is its range.
PARAMETER_RANGES = {
    'n_estimators': [30, 100, 300],
    'n_iterations': [1, 2, 3, 4],
    'dimension_mode': ['fixed', 'expanding'],
    'normalization': ['none', 'l1', 'l2'],
    'dimension_ratio': [1, 10, 20, 30, 40],
    'initial_projection_type': ['sparse', 'gaussian', 'signed_hash'],
    'path_projection_type': ['sparse', 'gaussian', 'signed_hash'],
    'concat_projection_type': ['sparse', 'gaussian', 'signed_hash'],
    'classifier': ['logistic_regression', 'random_forest', 'svm_rbf_auto_lambda'],
    'path_estimator': ['random_forest', 'recursive_partition'],
}

In [ ]:
search_outputs = run_search(
    dataset_names=DATASET_NAMES,
    dataset_count=DATASET_COUNT,
    max_dataset_size=MAX_DATASET_SIZE,
    dataset_seed=DATASET_SEED,
    repetitions=REPETITIONS,
    parameter_ranges=PARAMETER_RANGES,
    n_search_iter=N_SEARCH_ITER,
    search_seed=SEARCH_SEED,
    test_size=TEST_SIZE,
    n_jobs=N_JOBS,
    output_classifier_n_estimators=OUTPUT_CLASSIFIER_N_ESTIMATORS,
)
datasets = search_outputs['datasets']
dataset_summary = search_outputs['dataset_summary']
evaluation_blocks = search_outputs['evaluation_blocks']
sampled_parameters = search_outputs['sampled_parameters']
search_results = search_outputs['search_results']
block_results = search_outputs['block_results']

display(dataset_summary)
display(pd.DataFrame(sampled_parameters))
display(search_results)

In [ ]:
TOP_K = min(10, len(search_results))
top_results = search_results.head(TOP_K).copy()
display(top_results.style.format({
    'mean_accuracy': '{:.3f}',
    'std_dataset_accuracy': '{:.3f}',
    'mean_block_accuracy': '{:.3f}',
    'search_seconds': '{:.1f}',
}))

best_candidate_id, best_params = best_configuration(
    search_results,
    PARAMETER_RANGES,
)
display(pd.DataFrame([best_params]))
display(best_scores_by_dataset(block_results, best_candidate_id).style.format({
    'mean_accuracy': '{:.3f}',
    'std_accuracy': '{:.3f}',
}))

plot_top_candidates(search_results, top_k=TOP_K)
marginal_summary, marginal_figure = plot_marginalized_performance(
    search_results,
    PARAMETER_RANGES,
    best_params,
)
display(marginal_summary.style.format({
    'q25_accuracy': '{:.3f}',
    'median_accuracy': '{:.3f}',
    'q75_accuracy': '{:.3f}',
}))
plot_pairwise_performance(search_results, PARAMETER_RANGES, best_params)

The selected configuration maximizes the mean dataset-level accuracy on these evaluation blocks. For an unbiased final estimate, reserve additional datasets or repetitions as a validation set rather than reporting the same random-search score as final test performance.